In [ ]:
import pandas as pd
from collections import Counter
import yaml

In [ ]:
import os
os.chdir('../../../')

In [ ]:
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

### Import Chinese patents citation data

In [ ]:
citation_data = pd.read_stata(dataset_config['path_cnpat'] + '1985-2025/citation.dta')
citation_data.rename(columns={'申请号':'apn', '引证专利': 'citation'}, inplace=True)
citation_data

In [ ]:
def extract_country_codes(citation):
    if pd.isna(citation): 
        return ''
    parts = citation.split('; ')
    return ', '.join([part[:2] for part in parts])

In [ ]:
citation_data['cite_country'] = citation_data['citation'].apply(extract_country_codes)
citation_data = citation_data.drop(columns='citation')
citation_data

In [ ]:
# Define a function to count the number of citations per country
def count_countries(cite_country):
    countries = cite_country.split(', ')
    country_counts = Counter(countries)
    return country_counts

In [ ]:
# Apply this function to the cite_country column
country_counts_series = citation_data['cite_country'].apply(count_countries)

In [ ]:
country_df = pd.DataFrame(list(country_counts_series))
result_df = pd.concat([citation_data, country_df], axis=1).fillna(0)
empty_columns = result_df.columns[result_df.columns.str.strip() == '']
result_df = result_df.drop(columns=empty_columns)
result_df = result_df.drop(columns='cite_country')
result_df

In [ ]:
all_columns = result_df.columns.tolist()
columns_to_sum = [col for col in all_columns if col not in ['apn']]

domestic_columns_to_sum = [col for col in all_columns if col in ['CN']]
foriegn_columns_to_sum = [col for col in all_columns if col not in ['apn', 'CN', 'sum']]
us_columns_to_sum = [col for col in all_columns if col in ['US']]
nonUS_foriegn_columns_to_sum = [col for col in all_columns if col not in ['apn', 'CN', 'US', 'sum']]

# sum, foreign sum, non-US foreign sum
result_df['sum'] = result_df[columns_to_sum].sum(axis=1)
result_df['domestic_sum'] = result_df[domestic_columns_to_sum].sum(axis=1)
result_df['foreign_sum'] = result_df[foriegn_columns_to_sum].sum(axis=1)
result_df['us_sum'] = result_df[us_columns_to_sum].sum(axis=1)
result_df['nonUS_foreign_sum'] = result_df[nonUS_foriegn_columns_to_sum].sum(axis=1)


# foreign ratio, non-US foreign ratio
result_df['domestic_ratio'] = result_df['domestic_sum']/result_df['sum']
result_df['foreign_ratio'] = result_df['foreign_sum']/result_df['sum']
result_df['us_ratio'] = result_df['us_sum']/result_df['sum']
result_df['nonUS_foreign_ratio'] = result_df['nonUS_foreign_sum']/result_df['sum']

result_df = result_df.drop_duplicates()
result_df

In [ ]:
final_df = result_df[['apn', 'sum', 'domestic_sum', 'foreign_sum', 'us_sum',
                    'nonUS_foreign_sum', 'domestic_ratio', 'foreign_ratio',
                    'us_ratio', 'nonUS_foreign_ratio']]

In [ ]:
final_df.to_csv(dataset_config['path_processed'] + 'CN_CN/CNpat_citepat_country.csv', index=False)